# MPI direkt aus dem Notebook starten

Dieses Notebook zeigt, wie man MPI **aus dem Notebook heraus** mit `mpiexec` startet.

Das funktioniert nur, wenn MPI korrekt installiert ist.

Voraussetzung: 
- mpi-Umgebung auf dem Host installiert: `sudo apt install openmpi-bin libopenmpi-dev` (das können wir nicht nachholen)
- mpi4py installiert: `pip install mpi4py` (das wird zur Not hier nachgeholt)

In [18]:
pip install mpi4py

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Schreibe ein MPI-Skript
code = '''from mpi4py import MPI

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

print(f"Hello from process {rank} out of {size}")
'''

with open("hello_mpi.py", "w") as f:
    f.write(code)

print("hello_mpi.py erstellt")

hello_mpi.py erstellt


## MPI aus dem Notebook starten

Jetzt wird das Script mit mehreren Prozessen gestartet.


In [17]:
import os
import sys
import subprocess

# Pfad zu mpiexec muss ggfs je nach Umgebung angepasst werden.
cmd = [
    "/usr/bin/mpiexec",
    "--mca", "plm", "isolated",
    "-n", "4",
    sys.executable,
    "hello_mpi.py",
]

env = os.environ.copy()

if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
    env["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=env
)

# Nur die sinnvolle Programmausgabe zeigen
print(result.stdout)

# Warning nur anzeigen, wenn wirklich etwas anderes als der bekannte X11-Hinweis drinsteht
stderr_lines = [
    line for line in result.stderr.splitlines()
    if "Authorization required, but no authorization protocol specified" not in line
]

if stderr_lines:
    print("\nWeitere STDERR-Meldungen:")
    print("\n".join(stderr_lines))

print("\nReturncode:", result.returncode)

Hello from process 1 out of 4
Hello from process 0 out of 4
Hello from process 2 out of 4
Hello from process 3 out of 4


Weitere STDERR-Meldungen:







Returncode: 0


## Erwartete Ausgabe

Die Reihenfolge kann variieren:

```
Hello from process 0 out of 4
Hello from process 1 out of 4
Hello from process 2 out of 4
Hello from process 3 out of 4
```


## Troubleshooting

Wenn das nicht funktioniert:

- `mpiexec: command not found` → MPI nicht installiert
- `No module named mpi4py` → `pip install mpi4py`
